###  Section 1: Install & Setup

In [1]:
!pip install -U pageindex 

  Using cached pypdfium2-5.13.0-py3-none-win_amd64.whl.metadata (67 kB)
INFO: pip is looking at multiple versions of openai-agents to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/587.0 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/587.0 kB ? eta -:--:--
   ---------------------------------------- 587.0/587.0 kB 1.9 MB/s  0:00:00
   ---------------------------------------- 0.0/28.1 MB ? eta -:--:--
    --------------------------------------- 0.5/28.1 MB 2.7 MB/s eta 0:00:11
   - -------------------------------------- 1.0/28.1 MB 3.0 MB/s eta 0:00:09
   -- ------------------------------------- 1.8/28.1 MB 3.2 MB/s eta 0:00:09
   ---- ----------------------------------- 2.9/28.1 MB 3.6 MB/s eta 0:00:08
   ----- ---------------------------------- 3.9/28.1 MB 3.9 MB/s eta 0:00:07
   ------- -------------------------------- 5.0/28.1 MB 4.2 MB/s eta 0:00:06
   -------- -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [43]:
import os, json, time
from dotenv import load_dotenv
from mistralai.client import Mistral

load_dotenv()

PAGEINDEX_API_KEY = "017a520355f84b1486f1443fd52fbb28"
MISTRAL_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Mistral AI key loaded:", "✅" if MISTRAL_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Mistral AI key loaded: ✅


In [45]:
from pageindex import PageIndexClient
from mistralai.client import Mistral
from groq import Groq


pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
mistralai_client = Groq(api_key=MISTRAL_API_KEY)

print("✅ PageIndex client ready")
print("✅ mistral client ready")

✅ PageIndex client ready
✅ mistral client ready


### Section 2: Upload & Index a PDF

What happens here:

1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical tree index (like a smart Table of Contents)
4. Returns a doc_id for all future operations

Why NO chunking?
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [46]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./book.pdf"   

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"Uploaded!")
print(f"Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

Uploading: ./book.pdf
Uploaded!
Document ID: pi-cmunyy4rd00i709nzbwu9njyv
   (Save this ID — you'll use it throughout the notebook)


In [47]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\nTree index ready!")
        break
    elif status == "failed":
        print("\nProcessing failed. Check your PDF format.")
        break
    
    time.sleep(5)

Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

Tree index ready!


### 🔍 Section 3: Inspect the Tree Structure

Each node has:

    node_id — unique ID used during retrieval
    title — section heading
    page_index — page number in original PDF
    text — section summary (when node_summary=True)
    nodes — child sections (nested)

In [48]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\nRaw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 6

Raw tree (first node):
{
  "title": "Building Effective AI Agents: Architecture Patterns and Implementation Frameworks",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# Building Effective AI Agents: Architecture Patterns and Implementation Frameworks\n\nAccelerate your enterprise AI transformation with proven strategies from Anthropic's customers and internal teams.\n\nClaude\n\n## Contents\n\nExecutive summary 4\nCommon use cases and applications for AI agents 7\nCommon architecture patterns 10\nLooking forward: the future of building AI agents 27\nNext steps 29\n\n2\n\n![img-0.jpeg](img-0.jpeg)\n\nChapter 1\n",
  "text": "# Building Effective AI Agents: Architecture Patterns and Implementation Frameworks\n\nAccelerate your enterprise AI transformation with proven strategies from Anthropic's customers and internal teams.\n\nClaude\n\n## Contents\n\nExecutive summary 4\nCommon use cases and applications for AI agents 7\nCommon architecture patterns 10\nLo

In [49]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] Building Effective AI Agents: Architecture Patterns and Implementation Frameworks  (p.1)
[0001] Executive summary  (p.3)
[0002] Common use cases and applications for AI agents  (p.6)
[0003] Common architecture patterns  (p.9)
  └─ [0004] Agent design best practices  (p.10)
  └─ [0005] Single-agent systems  (p.11)
    └─ [0006] Example: Single-agent research agent  (p.12)
  └─ [0007] Multi-agent systems  (p.13)
    └─ [0008] Architecture patterns  (p.14)
    └─ [0009] Hierarchical/supervisory systems  (p.15)
    └─ [0010] Collaborative systems  (p.16)
  └─ [0011] Agentic workflows  (p.18)
    └─ [0012] Sequential workflows  (p.18)
    └─ [0013] Example: Multi-agent sequential workflow - automated data science insights  (p.19)
    └─ [0014] Parallel workflows  (p.20)
    └─ [0015] Example: Multi-agent parallel workflow - financial risk assessment  (p.20)
    └─ [0016] Evaluator-optimizer  (p.21)
    └─ [0017] Example: Multi-agent evaluator workflow - API 

In [50]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 28
   Each node = one retrievable section of the document


### Section 4: LLM Tree Search — The Core of PageIndex

This is where PageIndex fundamentally differs from vector RAG.

Vector RAG retrieval:
   - query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
1. Problem: finds what's similar, not what's relevant

PageIndex retrieval:
   - query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
2. Advantage: LLM understands document structure, context, and intent

#### The LLM acts like a human expert scanning a Table of Contents.

In [51]:
import json


def llm_tree_search(
    query: str,
    tree: list,
    model: str = "openai/gpt-oss-120b"
) -> dict:

    def compress(nodes):
        out = []

        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""
You are given a query and a document tree structure.

Your task is to identify which node IDs most likely contain
the information needed to answer the query.

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Return ONLY valid JSON in this format:

{{
  "thinking": "brief explanation of why the selected nodes are relevant",
  "node_list": ["node_id1", "node_id2"]
}}
"""

    response = mistralai_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    content = response.choices[0].message.content

    # Safety cleanup in case the model wraps JSON in ```json
    content = content.strip()

    if content.startswith("```json"):
        content = content[7:]

    if content.startswith("```"):
        content = content[3:]

    if content.endswith("```"):
        content = content[:-3]

    return json.loads(content.strip())

In [52]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in Agentic workflows ?"

print(f"Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

Query: What is the syllabus covered in Agentic workflows ?

LLM Reasoning:
The query asks for the syllabus (topics) covered under Agentic workflows. Node 0011 is the main section on Agentic workflows, and its child nodes detail the specific workflow types (Sequential, Parallel, Evaluator‑optimizer) and examples. These nodes together outline the curriculum for Agentic workflows.

Selected Node IDs: ['0011', '0012', '0014', '0016']
